# Full insight pipeline

Notebook объединяет выгрузку базового набора данных и последовательный запуск локального pipeline с промежуточными выводами.

Ожидаемая цепочка:

```text
выгрузка данных -> loaded_df -> enriched_df -> clustered_df -> filtered_df -> analyzed_df -> group_summaries
```

Перед запуском должны быть доступны `spark` и, если нужен MCC-справочник, `eng_pg`. API-ключи в notebook не используются.

In [ ]:
"""Standalone sequential notebook for data export and insight pipeline.

The notebook uses local pipeline classes. Put your local agent
and clustering library imports into the marked placeholder cell below.
"""

from __future__ import annotations

import json
import re
from collections import Counter
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Callable

import numpy as np
import pandas as pd
import pyspark.sql.types as T
from IPython.display import display
from pyspark.sql import DataFrame as SparkDataFrame
from pyspark.sql import functions as F

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 300)



## 1. Export parameters

Все таблицы и основные списки колонок вынесены в конфигурацию. Новые поля из `hits_extra` добавляй в `HITS_EXTRA_COLUMNS`, новые поля из `uko_event` - в `EVENT_EXTRA_COLUMNS` и при необходимости в `EVENT_STRUCT_COLUMNS`.

In [ ]:
@dataclass(frozen=True)
class ExportConfig:
    """Конфигурация выгрузки базового датасета.

    Args:
        date_from: Начальная дата целевого периода в формате YYYY-MM-DD.
        date_to: Конечная дата целевого периода в формате YYYY-MM-DD.
        digest_week: Номера недель дайджеста.
        history_weeks: Количество недель истории для расширенного окна.
        csi_table: Таблица CSI-ответов.
        hits_extra_table: Таблица расширенной информации по сработкам.
        automarking_table: Таблица истории автомаркировки.
        cards_event_table: Таблица card-событий.
        uko_event_table: Таблица UKO-событий.

    Returns:
        Объект с параметрами выгрузки.
    """

    date_from: str = "2026-03-28"
    date_to: str = "2026-04-10"
    digest_week: list[int] = field(default_factory=lambda: [14, 15])
    history_weeks: int = 6
    csi_table: str = "csp_repo_features3.history_csi_clients_answers_v2_129372114_129377108"
    hits_extra_table: str = "cspfs_repo_features3.hits_extra_info_129372427_view"
    automarking_table: str = "csp_repo_features.history_automarking_big_148078_155487"
    cards_event_table: str = "csp_afpc_sss_inc.cards_event"
    uko_event_table: str = "csp_afpc_sss_inc.uko_event"


EXPORT_CONFIG = ExportConfig()

HITS_EXTRA_COLUMNS = [
    "epk_id", "event_id", "event_dt", "event_channel", "sub_channel", "event_type", "age",
    "segment", "age_category", "resolution_first", "resolution_first_dttm",
    "resolution_last", "resolution_last_dttm", "surface", "atm_merchant_name",
    "merchant_info", "source_type_accept", "hits_extra_facts", "client_balance",
    "recipient_info", "scoring_oss", "previous_events_additional_info",
    "posterious_events_additional_info", "previous_events", "posterious_events",
]

CHANNEL_BY_SUB_CHANNEL = {
    "UFS.MOBILEAPI": "ДБО", "ISSUER_ACQUIRER": "Эмиссия", "UFS.BRANCHAPI": "ВСП",
    "ISSUER": "Эмиссия", "UFS.WEBAPI": "ДБО", "ESA.MOBILEAPI": "ДБО",
    "ISSUER_WEBACQUIRER": "Эмиссия", "ATMAPI": "ДБО", "UFS.MBKAPI": "ДБО",
    "UFS.ATMAPI": "ДБО", "UFS.OTHER": "ДБО", "ESA.WEBAPI": "ДБО",
    "ESA.BRANCHAPI": "ВСП",
}

MCC_NAME_OVERRIDES = {
    "6009": "Микрофинансовые организации", "3990": "Экосистема Яндекса",
    "3991": "Экосистема Сбера", "9390": "Госуслуги", "5262": "Маркетплейсы",
    "9406": "Микрофинансовые организации",
}

EVENT_EXTRA_COLUMNS = [
    "main_rule", "rules", "subrules", "list_geo_1km_user_id", "device_source_sdk",
    "user_ip_location_country_code", "user_ip_location_city",
    "payee_communication_term_with_ak_pf", "virus_names", "event_id",
]

EVENT_STRUCT_COLUMNS = [
    "evt_time", "event_type", "sub_type", "event_channel", "event_description",
    "transaction_amount", "atm_mcc", *EVENT_EXTRA_COLUMNS,
]

## 2. Shared helpers and date windows

In [ ]:
def show_df(df: pd.DataFrame, title: str, rows: int = 3) -> None:
    """Показывает состояние DataFrame после этапа.

    Args:
        df: DataFrame для вывода.
        title: Название этапа.
        rows: Количество строк для отображения.

    Returns:
        None. Функция печатает форму, колонки и отображает первые строки.
    """

    print(f"\n=== {title} ===")
    print(f"shape: {df.shape}")
    print(f"columns: {list(df.columns)}")
    display(df.head(rows))


def build_week_windows(config: ExportConfig) -> tuple[list[list[Any]], str, str, str]:
    """Рассчитывает недельные окна и технические даты.

    Args:
        config: Конфигурация выгрузки.

    Returns:
        Кортеж `(weeks_list, history_date_from, date_from_tst, date_to_tst)`.
    """

    date_to_ts = pd.to_datetime(config.date_to)
    current_week_start = (date_to_ts - pd.Timedelta(days=6)).strftime("%Y-%m-%d")
    weeks_list: list[list[Any]] = []
    for num in range(config.history_weeks, 0, -1):
        start_week = (pd.to_datetime(current_week_start) - pd.Timedelta(days=7 * num)).strftime("%Y-%m-%d")
        end_week = (date_to_ts - pd.Timedelta(days=7 * num)).strftime("%Y-%m-%d")
        weeks_list.append([start_week, end_week, config.digest_week[-1] - num])
    weeks_list.append([current_week_start, config.date_to, config.digest_week[-1]])
    history_date_from = weeks_list[0][0]
    date_from_tst = pd.to_datetime(history_date_from).strftime("%Y%m%d")
    date_to_tst = pd.to_datetime(config.date_to).strftime("%Y%m%d")
    return weeks_list, history_date_from, date_from_tst, date_to_tst


def safe_to_spark(pdf: pd.DataFrame) -> SparkDataFrame:
    """Преобразует pandas DataFrame в Spark DataFrame с явной схемой.

    Args:
        pdf: pandas DataFrame для конвертации.

    Returns:
        Spark DataFrame с предсказуемыми типами колонок.
    """

    prepared = pdf.copy()
    object_columns = prepared.select_dtypes(include=["object"]).columns
    prepared[object_columns] = prepared[object_columns].astype(str).replace("nan", None)
    fields: list[T.StructField] = []
    for column_name, dtype in prepared.dtypes.items():
        dtype_text = str(dtype).lower()
        if column_name == "epk_id":
            fields.append(T.StructField(column_name, T.LongType(), True))
        elif "int" in dtype_text:
            fields.append(T.StructField(column_name, T.LongType(), True))
        elif "float" in dtype_text:
            fields.append(T.StructField(column_name, T.DoubleType(), True))
        elif "datetime" in dtype_text:
            fields.append(T.StructField(column_name, T.TimestampType(), True))
        else:
            fields.append(T.StructField(column_name, T.StringType(), True))
    return spark.createDataFrame(prepared, schema=T.StructType(fields))


weeks_list, history_date_from, date_from_tst, date_to_tst = build_week_windows(EXPORT_CONFIG)
print("Целевой период:", EXPORT_CONFIG.date_from, EXPORT_CONFIG.date_to)
print("Историческое окно:", history_date_from, EXPORT_CONFIG.date_to)
display(pd.DataFrame(weeks_list, columns=["start_week", "end_week", "week_num"]))

## 3. CSI: load answers

In [ ]:
def normalize_type_accept(column: Any) -> Any:
    """Возвращает Spark-выражение для нормализации типа подтверждения.

    Args:
        column: Spark Column `type_accept`.

    Returns:
        Spark Column с нормализованным значением.
    """

    return (
        F.when(column == "ЕРКЦ", F.lit("ЕРКЦ"))
        .when(column == "HINT Эмиссия", F.lit("HINT Карты"))
        .when(column == "HINT ДБО", F.lit("HINT ДБО"))
        .when(column == "Сотрудник ВСП", F.lit("Сотрудник ВСП"))
        .when(column == "ГПМ", F.lit("ГПМ"))
        .when(column == "IVR", F.lit("IVR"))
        .when(column == "ChipPin", F.lit("Chip+Pin"))
        .when(column == "РО/ЗРО", F.lit("РО/ЗРО"))
        .when(column == "Black List", F.lit("Black List (БА)"))
        .when(column == "Сотрудник Бета", F.lit("Сотрудник Бета"))
        .when(column == "БИО", F.lit("BIO"))
        .when(column == "ДБ", F.lit("ДБ"))
        .when(column == "Подтверждение на близкого", F.lit("Подтверждение на близкого"))
        .when(column == "Сотрудник Гамма", F.lit("Сотрудник Гамма"))
        .otherwise(F.lit("Не определено/None"))
    )


def load_csi_answers(config: ExportConfig, history_start: str) -> SparkDataFrame:
    """Загружает CSI-ответы клиентов.

    Args:
        config: Конфигурация выгрузки.
        history_start: Начало исторического окна.

    Returns:
        Spark DataFrame с CSI-ответами.
    """

    return (
        spark.table(config.csi_table)
        .filter(F.col("answer_date").between(history_start, config.date_to))
        .withColumn("type_accept_corrected", normalize_type_accept(F.col("type_accept")))
    )


csi_raw_df = load_csi_answers(EXPORT_CONFIG, history_date_from).toPandas()
show_df(csi_raw_df, "2. CSI raw", rows=3)

## 4. CSI: pivot questions to wide format

In [ ]:
def transform_survey_data_clean(df: pd.DataFrame, group_columns: list[str] | None = None, question_count: int = 8) -> pd.DataFrame:
    """Разворачивает CSI-вопросы в широкий формат.

    Args:
        df: pandas DataFrame с CSI-ответами.
        group_columns: Колонки, задающие один кейс.
        question_count: Максимальное количество вопросов.

    Returns:
        pandas DataFrame с одной строкой на кейс.
    """

    if group_columns is None:
        group_columns = [column for column in ["epk_id", "event_id", "answer_date"] if column in df.columns]
    if not group_columns:
        raise ValueError("Нужна хотя бы одна колонка группировки для CSI.")
    result_rows: list[dict[str, Any]] = []
    question_columns = {"question_number", "question_text", "answer_text", "comment_text"}
    for _, group in df.groupby(group_columns, dropna=False):
        base_row: dict[str, Any] = {}
        for column in df.columns:
            if column in question_columns:
                continue
            values = group[column].dropna()
            base_row[column] = values.iloc[0] if not values.empty else None
        questions: dict[int, dict[str, Any]] = {}
        for _, row in group.iterrows():
            try:
                q_num = int(str(row.get("question_number")).strip())
            except (ValueError, TypeError):
                continue
            if 1 <= q_num <= question_count:
                questions[q_num] = {
                    "question": row.get("question_text"),
                    "answer": row.get("answer_text"),
                    "comment": row.get("comment_text"),
                }
        for q_num in range(1, question_count + 1):
            question_data = questions.get(q_num, {})
            base_row[f"question_{q_num}"] = question_data.get("question")
            base_row[f"answer_{q_num}"] = question_data.get("answer")
            base_row[f"comment_{q_num}"] = question_data.get("comment")
        result_rows.append(base_row)
    return pd.DataFrame(result_rows)


clients_hits_df = transform_survey_data_clean(csi_raw_df).drop(columns=["answer_8"], errors="ignore")
clients_hits_df["event_id"] = clients_hits_df["event_id"].astype(str)
show_df(clients_hits_df, "3. CSI wide / clients_hits_df", rows=3)

## 5. hits_extra: add base hit context

In [ ]:
def load_hits_extra(config: ExportConfig, event_ids: list[str], date_from_key: str, date_to_key: str) -> pd.DataFrame:
    """Загружает расширенную информацию по сработкам.

    Args:
        config: Конфигурация выгрузки.
        event_ids: Список event_id.
        date_from_key: Начальная дата в формате YYYYMMDD.
        date_to_key: Конечная дата в формате YYYYMMDD.

    Returns:
        pandas DataFrame с полями сработки.
    """

    if not event_ids:
        return pd.DataFrame(columns=HITS_EXTRA_COLUMNS)
    return (
        spark.read.table(config.hits_extra_table)
        .filter(F.col("event_dt").between(date_from_key, date_to_key))
        .filter(F.col("event_id").isin(event_ids))
        .select(*HITS_EXTRA_COLUMNS)
        .toPandas()
    )


event_ids_list = clients_hits_df["event_id"].dropna().astype(str).unique().tolist()
hits_extra_df = load_hits_extra(EXPORT_CONFIG, event_ids_list, date_from_tst, date_to_tst)
hits_extra_df["event_id"] = hits_extra_df["event_id"].astype(str)
show_df(hits_extra_df, "4. hits_extra_df", rows=3)

clients_hits_df = clients_hits_df.merge(hits_extra_df, on="event_id", how="left")
show_df(clients_hits_df, "4.1 after merge hits_extra", rows=3)

## 6. Clean duplicate columns and add channel

In [ ]:
def coalesce_duplicate_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Объединяет дублирующиеся колонки после merge.

    Args:
        df: DataFrame после merge.

    Returns:
        DataFrame без дублей имен колонок.
    """

    normalized = df.copy()
    normalized.columns = normalized.columns.str.replace("_x$", "", regex=True).str.replace("_y$", "", regex=True)
    normalized = normalized.replace("", np.nan)
    result: dict[str, pd.Series] = {}
    for column_name in normalized.columns.unique():
        columns = [normalized.iloc[:, idx] for idx, column in enumerate(normalized.columns) if column == column_name]
        merged = columns[0]
        for duplicate_column in columns[1:]:
            merged = merged.combine_first(duplicate_column)
        result[column_name] = merged
    return pd.DataFrame(result)


def add_business_channel(df: pd.DataFrame) -> pd.DataFrame:
    """Добавляет бизнес-канал по `sub_channel`.

    Args:
        df: DataFrame с колонкой `sub_channel`.

    Returns:
        DataFrame с колонкой `channel`.
    """

    result = df.copy()
    source = result["sub_channel"] if "sub_channel" in result.columns else pd.Series(index=result.index, dtype="object")
    result["channel"] = source.map(CHANNEL_BY_SUB_CHANNEL).fillna("Не определено")
    return result


clients_hits_df = coalesce_duplicate_columns(clients_hits_df)
clients_hits_df = add_business_channel(clients_hits_df)
show_df(clients_hits_df, "5. after coalesce + channel", rows=3)
display(clients_hits_df["channel"].value_counts(dropna=False).rename("count").reset_index())

## 7. MCC: load and enrich dictionary

In [ ]:
def load_atm_mcc(config: ExportConfig, event_ids: list[str], date_from_key: str, date_to_key: str) -> pd.DataFrame:
    """Загружает MCC по event_id.

    Args:
        config: Конфигурация выгрузки.
        event_ids: Список event_id.
        date_from_key: Начальная дата в формате YYYYMMDD.
        date_to_key: Конечная дата в формате YYYYMMDD.

    Returns:
        DataFrame с колонками `event_id`, `mcc_code`.
    """

    if not event_ids:
        return pd.DataFrame(columns=["event_id", "mcc_code"])
    return (
        spark.table(config.automarking_table)
        .withColumn("event_dt", F.date_format(F.col("event_time"), "yyyyMMdd"))
        .filter(F.col("event_dt").between(date_from_key, date_to_key))
        .filter(F.col("event_id").isin(event_ids))
        .selectExpr("event_id", "cast(atm_mcc as string) as mcc_code")
        .toPandas()
    )


def load_mcc_dictionary() -> pd.DataFrame:
    """Загружает справочник MCC.

    Args:
        Нет входных аргументов. Используется внешний объект `eng_pg`.

    Returns:
        DataFrame с колонками `mcc_code`, `mcc_name`, `description`.
    """

    mcc_request = """
    SELECT a.*, b.*
    FROM selfdags.mcc_codes a
    LEFT JOIN dashboards.mcc_dictionary b
    ON CAST(a.mcc_code AS INT) = CAST(b.mcc AS INT)
    """
    mcc_pd = pd.read_sql(mcc_request, eng_pg).drop(columns=["mcc", "activity"], errors="ignore")
    mcc_pd = mcc_pd[["mcc_code", "mcc_name", "description"]].drop_duplicates()
    mcc_pd["mcc_code"] = mcc_pd["mcc_code"].astype(str)
    return mcc_pd


def enrich_mcc_names(df: pd.DataFrame, mcc_dictionary: pd.DataFrame) -> pd.DataFrame:
    """Добавляет описание MCC и бизнес-переопределения названий.

    Args:
        df: DataFrame с колонкой `mcc_code`.
        mcc_dictionary: Справочник MCC.

    Returns:
        DataFrame с `mcc_name` и `description`.
    """

    result = df.copy()
    result["mcc_code"] = result["mcc_code"].astype(str)
    result = result.merge(mcc_dictionary, on="mcc_code", how="left")
    for mcc_code, mcc_name in MCC_NAME_OVERRIDES.items():
        result["mcc_name"] = np.where(result["mcc_code"] == mcc_code, mcc_name, result["mcc_name"])
    return result


atm_mcc_df = load_atm_mcc(EXPORT_CONFIG, event_ids_list, date_from_tst, date_to_tst)
atm_mcc_df["event_id"] = atm_mcc_df["event_id"].astype(str)
show_df(atm_mcc_df, "6. atm_mcc_df", rows=3)

clients_hits_df = clients_hits_df.merge(atm_mcc_df, on="event_id", how="left")
mcc_dictionary_df = load_mcc_dictionary()
show_df(mcc_dictionary_df, "6.1 mcc_dictionary_df", rows=3)

clients_hits_df = enrich_mcc_names(clients_hits_df, mcc_dictionary_df)
show_df(clients_hits_df, "6.2 after MCC enrichment", rows=3)

## 8. Client events on hit day

In [ ]:
def empty_events_schema() -> T.StructType:
    """Создает схему пустого DataFrame для событий.

    Args:
        Нет входных аргументов.

    Returns:
        Spark StructType для unionByName.
    """

    base_fields = [
        T.StructField("epk_id", T.LongType(), True),
        T.StructField("event_dt", T.StringType(), True),
        T.StructField("event_type", T.StringType(), True),
        T.StructField("sub_type", T.StringType(), True),
        T.StructField("event_channel", T.StringType(), True),
        T.StructField("event_description", T.StringType(), True),
        T.StructField("transaction_amount", T.StringType(), True),
        T.StructField("atm_mcc", T.StringType(), True),
        T.StructField("evt_time", T.TimestampType(), True),
    ]
    return T.StructType(base_fields + [T.StructField(column, T.StringType(), True) for column in EVENT_EXTRA_COLUMNS])


def load_events_for_source(table_name: str, keys: SparkDataFrame, has_extra: bool) -> SparkDataFrame:
    """Загружает события за день по точным парам `epk_id`, `event_dt`.

    Args:
        table_name: Название таблицы событий.
        keys: Spark DataFrame с ключами `epk_id`, `event_dt`.
        has_extra: Есть ли в source-таблице UKO extra-поля.

    Returns:
        Spark DataFrame с унифицированной схемой событий.
    """

    if keys.rdd.isEmpty():
        return spark.createDataFrame([], schema=empty_events_schema())
    source = (
        spark.read.table(table_name)
        .withColumn("epk_id", F.col("epk_id").cast("long"))
        .withColumn("event_dt", F.col("event_dt").cast("string"))
    )
    joined = source.join(keys, on=["epk_id", "event_dt"], how="inner")
    select_list = [
        F.col("epk_id").cast("long").alias("epk_id"),
        F.col("event_dt").cast("string").alias("event_dt"),
        F.col("event_type").cast("string").alias("event_type"),
        F.col("sub_type").cast("string").alias("sub_type"),
        F.col("event_channel").cast("string").alias("event_channel"),
        F.col("event_description").cast("string").alias("event_description"),
        F.col("transaction_amount").cast("string").alias("transaction_amount"),
        F.col("atm_mcc").cast("string").alias("atm_mcc"),
        F.from_unixtime(F.col("event_time").cast("long") / 1000).cast("timestamp").alias("evt_time"),
    ]
    if has_extra:
        select_list.extend([F.col(column).cast("string").alias(column) for column in EVENT_EXTRA_COLUMNS])
    else:
        select_list.extend([F.lit(None).cast("string").alias(column) for column in EVENT_EXTRA_COLUMNS])
    return joined.select(*select_list)


def load_day_events(config: ExportConfig, cases_pdf: pd.DataFrame) -> pd.DataFrame:
    """Добавляет события клиента за день сработки.

    Args:
        config: Конфигурация выгрузки.
        cases_pdf: Базовый DataFrame с кейсами.

    Returns:
        DataFrame с колонкой `events_day`.
    """

    hits = safe_to_spark(cases_pdf)
    base_keys = hits.select(F.col("epk_id").cast("long").alias("epk_id"), F.col("event_dt").cast("string").alias("event_dt"), "channel")
    cards_keys = base_keys.filter(F.col("channel") == "Эмиссия").select("epk_id", "event_dt").distinct()
    uko_keys = base_keys.filter((F.col("channel") != "Эмиссия") | F.col("channel").isNull()).select("epk_id", "event_dt").distinct()
    events_cards = load_events_for_source(config.cards_event_table, cards_keys, has_extra=False)
    events_uko = load_events_for_source(config.uko_event_table, uko_keys, has_extra=True)
    all_events = events_cards.unionByName(events_uko)
    events_col = all_events.groupBy("epk_id", "event_dt").agg(F.sort_array(F.collect_list(F.struct(*EVENT_STRUCT_COLUMNS))).alias("events_day"))
    result = hits.join(events_col, on=["epk_id", "event_dt"], how="left")
    for column in ["event_time", "answer_date", "answer_time", "first_resolution_time", "last_resolution_time"]:
        if column in result.columns:
            result = result.withColumn(column, F.date_format(F.col(column), "yyyy-MM-dd HH:mm:ss"))
    return result.toPandas()


result_pandas = load_day_events(EXPORT_CONFIG, clients_hits_df)
show_df(result_pandas, "7. result_pandas with events_day", rows=3)

## 9. Local pipeline classes

Standalone classes used directly inside this notebook.


In [ ]:
class DumpMixin:
    """Mixin with Pydantic-like model_dump for local dataclasses."""

    def model_dump(self, mode: str | None = None) -> dict[str, Any]:
        """Return dataclass fields as a dictionary."""
        _ = mode
        return asdict(self)


@dataclass
class MissingDataRequest(DumpMixin):
    """Request for additional data returned by the case agent."""

    source_name: str = ""
    reason: str = ""
    lookup_keys: dict[str, Any] = field(default_factory=dict)
    period: str | None = None
    priority: str = "medium"


@dataclass
class CaseAnalysisRecord(DumpMixin):
    """Result of processing one row with the case agent."""

    row_index: int
    case_id: str = ""
    group_name: str = ""
    status: str = "processed"
    agent_prompt: str = ""
    report_markdown: str = ""
    structured_result: dict[str, Any] = field(default_factory=dict)
    missing_data_requests: list[MissingDataRequest] = field(default_factory=list)
    error: str = ""


@dataclass
class GroupSelectionDecision(DumpMixin):
    """Binary decision about whether a group should be analyzed."""

    group_name: str
    is_meaningful: bool
    reason: str = ""
    confidence: float = 0.0
    total_records: int = 0


@dataclass
class GroupProblemSummary(DumpMixin):
    """Final summary for one selected group."""

    group_name: str
    total_records: int = 0
    processed_records: int = 0
    failed_records: int = 0
    missing_data_requests_count: int = 0
    problem_summary: str = ""
    evidence_case_ids: list[str] = field(default_factory=list)
    limitations: list[str] = field(default_factory=list)


@dataclass
class InsightPipelineConfig(DumpMixin):
    """Configuration for the insight pipeline."""

    text_column: str = "comment_text"
    group_column: str = "problem_group"
    case_id_column: str = "case_id"
    event_id_column: str = "event_id"
    selected_group_column: str = "is_significant_group"
    group_selection_reason_column: str = "group_selection_reason"
    agent_report_column: str = "agent_record_description"
    agent_status_column: str = "agent_processing_status"
    agent_error_column: str = "agent_processing_error"
    agent_structured_result_column: str = "agent_structured_result"
    agent_missing_data_requests_column: str = "agent_missing_data_requests"
    max_cases_per_group: int | None = None
    min_group_size: int = 3
    max_groups: int | None = 10
    agent_recursion_limit: int = 60
    include_full_row_prompt: bool = True


class NoopDataEnricher:
    """No-op enrichment component."""

    def enrich(self, df: pd.DataFrame) -> pd.DataFrame:
        """Return a copy of the input DataFrame."""
        return df.copy()


class StubCommentClusterer:
    """Placeholder clusterer. Replace it with your clustering library."""

    def cluster(self, df: pd.DataFrame, *, text_column: str, group_column: str) -> pd.DataFrame:
        """Return DataFrame with a group column."""
        _ = text_column
        result = df.copy()
        if group_column not in result.columns:
            result[group_column] = "Без группы"
        result[group_column] = result[group_column].fillna("Без группы").astype(str)
        return result


class StubSignificantGroupSelector:
    """Placeholder binary group selector instead of LLM."""

    def classify_group(self, group_name: str, group_df: pd.DataFrame, config: InsightPipelineConfig) -> GroupSelectionDecision:
        """Classify one group by deterministic size rule."""
        total_records = len(group_df)
        is_meaningful = total_records >= config.min_group_size
        sign = ">=" if is_meaningful else "<"
        return GroupSelectionDecision(
            group_name=group_name,
            is_meaningful=is_meaningful,
            reason=f"Размер группы {total_records} {sign} min_group_size={config.min_group_size}.",
            confidence=1.0,
            total_records=total_records,
        )


def _safe_value_to_text(value: Any) -> str:
    """Convert a cell value to prompt-safe text."""
    if value is None:
        return "Значение отсутствует"
    try:
        if isinstance(value, float) and np.isnan(value):
            return "Значение отсутствует"
    except TypeError:
        pass
    return str(value)


def build_case_prompt(row: dict[str, Any], *, group_name: str, row_index: int, max_string_length: int | None = None) -> str:
    """Build a plain text request for the agent from one DataFrame row."""
    lines = [
        "Разбери одну антифрод-сработку клиента.",
        "",
        f"Индекс строки: {row_index}",
        f"Предварительная группа: {group_name}",
        "",
        "Данные строки:",
    ]
    for key, value in row.items():
        text = _safe_value_to_text(value)
        if max_string_length is not None and len(text) > max_string_length:
            text = text[:max_string_length] + "..."
        lines.append(f"- {key}: {text}")
    lines.extend([
        "",
        "Задача:",
        "1. Реши, хватает ли данных для описания сработки.",
        "2. Если данных не хватает, явно перечисли, какие данные нужно дозагрузить.",
        "3. Если данных хватает, напиши краткое фактическое описание записи.",
        "4. Если возвращаешь JSON, помести его в теги <structured_result>...</structured_result>.",
    ])
    return "\n".join(lines)


def _content_to_text(response: Any) -> str:
    """Extract text from agent response."""
    if isinstance(response, str):
        return response
    if isinstance(response, list) and response:
        last_item = response[-1]
        content = getattr(last_item, "content", None)
        return str(content) if content is not None else str(last_item)
    content = getattr(response, "content", None)
    return str(content) if content is not None else str(response)


def _extract_structured_result(text: str) -> dict[str, Any]:
    """Extract JSON structured result from agent response."""
    tag_match = re.search(r"<structured_result>\s*(?P<payload>.*?)\s*</structured_result>", text, flags=re.DOTALL | re.IGNORECASE)
    payload = tag_match.group("payload").strip() if tag_match else None
    if payload is None:
        fenced_match = re.search(r"json\s*(?P<payload>\{.*?\})", text, flags=re.DOTALL | re.IGNORECASE)
        payload = fenced_match.group("payload") if fenced_match else None
    if payload is None:
        return {}
    try:
        return json.loads(payload)
    except Exception:
        return {}


def _build_case_id(row: pd.Series, config: InsightPipelineConfig, row_index: int) -> str:
    """Build case identifier for one row."""
    if config.case_id_column in row and pd.notna(row[config.case_id_column]):
        return str(row[config.case_id_column])
    if config.event_id_column in row and pd.notna(row[config.event_id_column]):
        return str(row[config.event_id_column])
    return f"row_{row_index}"


class CaseAgentProcessor:
    """Adapter that runs the single-case agent for DataFrame rows."""

    def __init__(self, agent: Any | None = None, *, prompt_builder: Callable[..., str] = build_case_prompt, max_string_length: int | None = None) -> None:
        self.agent = agent
        self.prompt_builder = prompt_builder
        self.max_string_length = max_string_length

    def process_row(self, row: pd.Series, *, row_index: int, group_name: str, config: InsightPipelineConfig) -> CaseAnalysisRecord:
        """Process one row with the agent."""
        case_id = _build_case_id(row, config, row_index)
        prompt = self.prompt_builder(row.to_dict(), group_name=group_name, row_index=row_index, max_string_length=self.max_string_length)
        if self.agent is None:
            return CaseAnalysisRecord(row_index=row_index, case_id=case_id, group_name=group_name, status="skipped", agent_prompt=prompt, report_markdown="Агент не подключен. Сформирован только prompt для будущей обработки.")
        try:
            response = self.agent.invoke(prompt, config={"recursion_limit": config.agent_recursion_limit})
            report_text = _content_to_text(response)
            structured_result = _extract_structured_result(report_text)
            missing_data_requests = [MissingDataRequest(**item) for item in structured_result.get("missing_data_requests", []) if isinstance(item, dict)]
            return CaseAnalysisRecord(row_index=row_index, case_id=case_id, group_name=group_name, status="processed", agent_prompt=prompt, report_markdown=report_text, structured_result=structured_result, missing_data_requests=missing_data_requests)
        except Exception as exc:
            return CaseAnalysisRecord(row_index=row_index, case_id=case_id, group_name=group_name, status="failed", agent_prompt=prompt, error=str(exc))


class GroupProblemSummarizer:
    """Summarizer for selected problem groups."""

    def __init__(self, summarizer: Any | None = None) -> None:
        self.summarizer = summarizer

    def summarize(self, df: pd.DataFrame, records: list[CaseAnalysisRecord], *, selected_groups: list[str], config: InsightPipelineConfig) -> list[GroupProblemSummary]:
        """Build final summaries for selected groups."""
        records_by_group: dict[str, list[CaseAnalysisRecord]] = {}
        for record in records:
            records_by_group.setdefault(record.group_name, []).append(record)
        summaries = []
        for group_name in selected_groups:
            group_df = df[df[config.group_column].astype(str) == group_name]
            group_records = records_by_group.get(group_name, [])
            if self.summarizer is not None:
                summaries.append(self._summarize_with_external(group_name, group_df, group_records))
            else:
                summaries.append(self._summarize_deterministic(group_name, group_df, group_records))
        return summaries

    def _summarize_deterministic(self, group_name: str, group_df: pd.DataFrame, records: list[CaseAnalysisRecord]) -> GroupProblemSummary:
        """Build deterministic group summary without LLM."""
        processed = [record for record in records if record.status == "processed"]
        failed = [record for record in records if record.status == "failed"]
        missing_count = sum(len(record.missing_data_requests) for record in records)
        summaries = [str(record.structured_result.get("final_summary", "")).strip() for record in processed if record.structured_result.get("final_summary")]
        if summaries:
            problem_summary = Counter(summaries).most_common(1)[0][0]
        elif processed:
            problem_summary = processed[0].report_markdown[:700]
        else:
            problem_summary = "Группа выбрана для анализа, но агентные отчеты пока отсутствуют или не были успешно обработаны."
        limitations = []
        if missing_count:
            limitations.append(f"Агент запросил дозагрузку данных: {missing_count} запросов.")
        if failed:
            limitations.append(f"Ошибки обработки кейсов: {len(failed)}.")
        if len(processed) < len(group_df):
            limitations.append(f"Разобрано {len(processed)} из {len(group_df)} строк группы; вывод ограничен покрытием.")
        return GroupProblemSummary(group_name=group_name, total_records=int(len(group_df)), processed_records=len(processed), failed_records=len(failed), missing_data_requests_count=missing_count, problem_summary=problem_summary, evidence_case_ids=[record.case_id for record in processed[:5]], limitations=limitations)

    def _summarize_with_external(self, group_name: str, group_df: pd.DataFrame, records: list[CaseAnalysisRecord]) -> GroupProblemSummary:
        """Build group summary through an external summarizer."""
        payload = {"group_name": group_name, "total_records": len(group_df), "case_reports": [record.model_dump() for record in records]}
        raw_summary = self.summarizer(payload) if callable(self.summarizer) else self.summarizer.invoke(payload)
        return GroupProblemSummary(group_name=group_name, total_records=int(len(group_df)), processed_records=len([record for record in records if record.status == "processed"]), failed_records=len([record for record in records if record.status == "failed"]), missing_data_requests_count=sum(len(record.missing_data_requests) for record in records), problem_summary=_content_to_text(raw_summary), evidence_case_ids=[record.case_id for record in records[:5]], limitations=[])


@dataclass
class InsightPipelineRun(DumpMixin):
    """Container with all intermediate and final pipeline results."""

    loaded_df: pd.DataFrame
    enriched_df: pd.DataFrame
    clustered_df: pd.DataFrame
    filtered_df: pd.DataFrame
    analyzed_df: pd.DataFrame
    group_selection_decisions: list[GroupSelectionDecision] = field(default_factory=list)
    selected_groups: list[str] = field(default_factory=list)
    case_results: list[CaseAnalysisRecord] = field(default_factory=list)
    group_summaries: list[GroupProblemSummary] = field(default_factory=list)
    metadata: dict[str, Any] = field(default_factory=dict)

    def case_results_df(self) -> pd.DataFrame:
        """Return case results as DataFrame."""
        return pd.DataFrame([record.model_dump() for record in self.case_results])

    def group_summaries_df(self) -> pd.DataFrame:
        """Return group summaries as DataFrame."""
        return pd.DataFrame([summary.model_dump() for summary in self.group_summaries])

    def group_selection_decisions_df(self) -> pd.DataFrame:
        """Return group selection decisions as DataFrame."""
        return pd.DataFrame([decision.model_dump() for decision in self.group_selection_decisions])


class InsightPipeline:
    """Full insight pipeline over a single-case agent."""

    def __init__(self, *, config: InsightPipelineConfig | None = None, enricher: Any | None = None, clusterer: Any | None = None, group_selector: Any | None = None, case_processor: CaseAgentProcessor | None = None, group_summarizer: GroupProblemSummarizer | None = None) -> None:
        self.config = config or InsightPipelineConfig()
        self.enricher = enricher or NoopDataEnricher()
        self.clusterer = clusterer or StubCommentClusterer()
        self.group_selector = group_selector or StubSignificantGroupSelector()
        self.case_processor = case_processor or CaseAgentProcessor(agent=None)
        self.group_summarizer = group_summarizer or GroupProblemSummarizer()

    def _cluster(self, df: pd.DataFrame) -> pd.DataFrame:
        """Run comment clustering."""
        if callable(getattr(self.clusterer, "cluster", None)):
            return self.clusterer.cluster(df, text_column=self.config.text_column, group_column=self.config.group_column)
        if callable(self.clusterer):
            return self.clusterer(df, text_column=self.config.text_column, group_column=self.config.group_column)
        raise TypeError("clusterer must have cluster(...) method or be callable.")

    def _classify_groups(self, df: pd.DataFrame) -> list[GroupSelectionDecision]:
        """Classify each group as meaningful or not meaningful."""
        normalized_df = df.copy()
        normalized_df[self.config.group_column] = normalized_df[self.config.group_column].fillna("Без группы").astype(str)
        decisions = []
        for group_name, group_df in normalized_df.groupby(self.config.group_column, dropna=False):
            if callable(getattr(self.group_selector, "classify_group", None)):
                raw_decision = self.group_selector.classify_group(str(group_name), group_df, self.config)
            elif callable(self.group_selector):
                raw_decision = self.group_selector(str(group_name), group_df, self.config)
            else:
                raise TypeError("group_selector must have classify_group(...) method or be callable.")
            decisions.append(raw_decision if isinstance(raw_decision, GroupSelectionDecision) else GroupSelectionDecision(**raw_decision))
        return self._apply_group_limit(decisions)

    def _apply_group_limit(self, decisions: list[GroupSelectionDecision]) -> list[GroupSelectionDecision]:
        """Apply max_groups limit to positive decisions."""
        if self.config.max_groups is None:
            return decisions
        kept_count = 0
        limited = []
        for decision in decisions:
            if not decision.is_meaningful:
                limited.append(decision)
                continue
            kept_count += 1
            if kept_count <= self.config.max_groups:
                limited.append(decision)
            else:
                limited.append(GroupSelectionDecision(group_name=decision.group_name, is_meaningful=False, reason=f"{decision.reason} Группа отклонена из-за max_groups={self.config.max_groups}.", confidence=decision.confidence, total_records=decision.total_records))
        return limited

    def _filter_selected_groups(self, df: pd.DataFrame, decisions: list[GroupSelectionDecision]) -> pd.DataFrame:
        """Filter DataFrame to selected groups only."""
        selected_groups = {decision.group_name for decision in decisions if decision.is_meaningful}
        reason_by_group = {decision.group_name: decision.reason for decision in decisions}
        result = df.copy()
        result[self.config.group_column] = result[self.config.group_column].fillna("Без группы").astype(str)
        result[self.config.selected_group_column] = result[self.config.group_column].isin(selected_groups)
        result[self.config.group_selection_reason_column] = result[self.config.group_column].map(reason_by_group).fillna("")
        return result[result[self.config.selected_group_column]].copy()

    def _process_filtered_cases(self, df: pd.DataFrame) -> tuple[pd.DataFrame, list[CaseAnalysisRecord]]:
        """Process filtered rows through the single-case agent."""
        analyzed_df = df.copy()
        analyzed_df[self.config.agent_report_column] = ""
        analyzed_df[self.config.agent_status_column] = ""
        analyzed_df[self.config.agent_error_column] = ""
        analyzed_df[self.config.agent_structured_result_column] = None
        analyzed_df[self.config.agent_missing_data_requests_column] = None
        records = []
        for group_name, group_df in analyzed_df.groupby(self.config.group_column, dropna=False):
            group_rows = group_df if self.config.max_cases_per_group is None else group_df.head(self.config.max_cases_per_group)
            for row_index, row in group_rows.iterrows():
                record = self.case_processor.process_row(row, row_index=int(row_index), group_name=str(group_name), config=self.config)
                records.append(record)
                analyzed_df.at[row_index, self.config.agent_report_column] = record.report_markdown
                analyzed_df.at[row_index, self.config.agent_status_column] = record.status
                analyzed_df.at[row_index, self.config.agent_error_column] = record.error
                analyzed_df.at[row_index, self.config.agent_structured_result_column] = record.structured_result
                analyzed_df.at[row_index, self.config.agent_missing_data_requests_column] = [request.model_dump() for request in record.missing_data_requests]
        return analyzed_df, records



## 10. External agent and clustering placeholders

Replace this cell with imports from your neighboring agent folder and your clustering library.


In [ ]:
# === PLACE FOR YOUR REAL IMPORTS ===
# Agent from a neighboring folder:
# import sys
# sys.path.append("../path_to_agent_project")
# from your_agent_module import research_agent

# Clustering library installed as a package:
# from your_clustering_library import comment_clusterer

# Optional LLM group selector and group summarizer:
# from your_group_selector_library import group_selector
# from your_group_summarizer_library import group_summarizer

research_agent = globals().get("research_agent", None)
comment_clusterer = globals().get("comment_clusterer", None)
group_selector = globals().get("group_selector", None)
group_summarizer = globals().get("group_summarizer", None)

print("research_agent:", type(research_agent).__name__ if research_agent is not None else None)
print("comment_clusterer:", type(comment_clusterer).__name__ if comment_clusterer is not None else None)
print("group_selector:", type(group_selector).__name__ if group_selector is not None else None)
print("group_summarizer:", type(group_summarizer).__name__ if group_summarizer is not None else None)



## 11. Pipeline component settings

In [ ]:
PIPELINE_CONFIG = InsightPipelineConfig(
    text_column="comment_1" if "comment_1" in result_pandas.columns else "comment_text",
    group_column="problem_group",
    case_id_column="case_id" if "case_id" in result_pandas.columns else "event_id",
    event_id_column="event_id",
    min_group_size=3,
    max_groups=10,
    max_cases_per_group=None,
    agent_recursion_limit=60,
)

clusterer = comment_clusterer if comment_clusterer is not None else StubCommentClusterer()
selector = group_selector if group_selector is not None else StubSignificantGroupSelector()
agent = research_agent

pipeline = InsightPipeline(
    config=PIPELINE_CONFIG,
    enricher=NoopDataEnricher(),
    clusterer=clusterer,
    group_selector=selector,
    case_processor=CaseAgentProcessor(agent=agent, max_string_length=None),
    group_summarizer=GroupProblemSummarizer(summarizer=group_summarizer),
)

display(PIPELINE_CONFIG.model_dump())
print("clusterer:", type(clusterer).__name__)
print("selector:", type(selector).__name__)
print("agent:", type(agent).__name__ if agent is not None else None)



## 12. Pipeline step 1-2: loaded_df and enriched_df

In [ ]:
loaded_df = result_pandas.copy()
show_df(loaded_df, "9.1 loaded_df", rows=3)

enriched_df = pipeline.enricher.enrich(loaded_df)
show_df(enriched_df, "9.2 enriched_df", rows=3)

## 13. Pipeline step 3: clustered_df

In [ ]:
clustered_df = pipeline._cluster(enriched_df)
show_df(clustered_df, "10. clustered_df", rows=3)
display(clustered_df[PIPELINE_CONFIG.group_column].value_counts(dropna=False).head(20).rename("group_size").reset_index())

## 14. Pipeline step 4: binary group selection

In [ ]:
group_selection_decisions = pipeline._classify_groups(clustered_df)
group_selection_decisions_df = pd.DataFrame([decision.model_dump(mode="json") for decision in group_selection_decisions])
show_df(group_selection_decisions_df, "11. group_selection_decisions", rows=20)

## 15. Pipeline step 5: filtered_df

In [ ]:
filtered_df = pipeline._filter_selected_groups(clustered_df, group_selection_decisions)
show_df(filtered_df, "12. filtered_df", rows=3)
display(filtered_df[PIPELINE_CONFIG.group_column].value_counts(dropna=False).head(20).rename("filtered_group_size").reset_index())

## 16. Pipeline step 6: analyzed_df + case_results

In [ ]:
analyzed_df, case_results = pipeline._process_filtered_cases(filtered_df)
show_df(analyzed_df, "13. analyzed_df", rows=3)

case_results_df = pd.DataFrame([record.model_dump(mode="json") for record in case_results])
show_df(case_results_df, "13.1 case_results_df", rows=3)

## 17. Pipeline step 7: group_summaries

In [ ]:
selected_groups = [decision.group_name for decision in group_selection_decisions if decision.is_meaningful]
group_summaries = pipeline.group_summarizer.summarize(
    analyzed_df,
    case_results,
    selected_groups=selected_groups,
    config=PIPELINE_CONFIG,
)
group_summaries_df = pd.DataFrame([summary.model_dump(mode="json") for summary in group_summaries])
show_df(group_summaries_df, "14. group_summaries_df", rows=20)

## 18. Build InsightPipelineRun

In [ ]:
pipeline_run = InsightPipelineRun(
    loaded_df=loaded_df,
    enriched_df=enriched_df,
    clustered_df=clustered_df,
    filtered_df=filtered_df,
    analyzed_df=analyzed_df,
    group_selection_decisions=group_selection_decisions,
    selected_groups=selected_groups,
    case_results=case_results,
    group_summaries=group_summaries,
    metadata={
        "loaded_rows": len(loaded_df),
        "enriched_rows": len(enriched_df),
        "clustered_rows": len(clustered_df),
        "filtered_rows": len(filtered_df),
        "analyzed_rows": len(analyzed_df),
        "selected_groups_count": len(selected_groups),
        "case_results_count": len(case_results),
    },
)

display(pipeline_run.metadata)
display(pipeline_run.selected_groups)

## 19. Final markdown report by groups

In [ ]:
def build_groups_markdown_report(summaries_df: pd.DataFrame) -> str:
    """Формирует markdown-отчет по итоговым групповым сводкам.

    Args:
        summaries_df: DataFrame из `pipeline_run.group_summaries_df()`.

    Returns:
        Markdown-строка с итогами по группам.
    """

    parts = ["# Итоговая суммаризация проблемных групп\n"]
    for _, row in summaries_df.iterrows():
        parts.append(f"## {row['group_name']}")
        parts.append(f"- Всего записей: {row['total_records']}")
        parts.append(f"- Обработано агентом: {row['processed_records']}")
        parts.append(f"- Ошибок обработки: {row['failed_records']}")
        parts.append(f"- Запросов на дозагрузку: {row['missing_data_requests_count']}")
        parts.append("")
        parts.append(str(row["problem_summary"]))
        parts.append("")
        limitations = row.get("limitations", [])
        if limitations:
            parts.append("Ограничения:")
            for item in limitations:
                parts.append(f"- {item}")
            parts.append("")
    return "\n".join(parts)


final_report_md = build_groups_markdown_report(group_summaries_df)
print(final_report_md)

## 20. Optional result saving

In [ ]:
# output_dir = Path("pipeline_outputs")
# output_dir.mkdir(exist_ok=True)
# loaded_df.to_parquet(output_dir / "01_loaded_df.parquet", index=False)
# enriched_df.to_parquet(output_dir / "02_enriched_df.parquet", index=False)
# clustered_df.to_parquet(output_dir / "03_clustered_df.parquet", index=False)
# filtered_df.to_parquet(output_dir / "04_filtered_df.parquet", index=False)
# analyzed_df.to_parquet(output_dir / "05_analyzed_df.parquet", index=False)
# group_selection_decisions_df.to_csv(output_dir / "group_selection_decisions.csv", index=False, encoding="utf-8-sig")
# case_results_df.to_json(output_dir / "case_results.jsonl", orient="records", lines=True, force_ascii=False)
# group_summaries_df.to_csv(output_dir / "group_summaries.csv", index=False, encoding="utf-8-sig")
# (output_dir / "final_report.md").write_text(final_report_md, encoding="utf-8")